# Detector de Pastillas — Cámara en Vivo (Sin ML)

**Flujo de dos fases:**
1. **Calibración automática** — lee las imágenes de referencia de `pills/`, elimina el fondo por muestreo de bordes y construye un perfil de histograma HSV + rasgos de forma para cada pastilla.
2. **Detección en cámara** — captura video de la webcam, segmenta lo que haya en el recuadro central y lo compara con los perfiles calibrados usando correlación de histogramas y similitud de forma.

In [33]:
import cv2
import numpy as np
import os
import glob

## Fase 1 — Calibración automática desde `pills/`

Para cada imagen de referencia:
- Se elimina el fondo muestreando los píxeles del borde de la imagen (sin suponer un color fijo de fondo).
- Se encuentra el contorno más grande → la pastilla.
- Se calculan histogramas HSV normalizados (canales H, S y V por separado).
- Se guardan el **aspect ratio** y la **solidez** del contorno como descriptores de forma.

In [34]:
PILLS_FOLDER = 'pills'
DISPLAY_SIZE = 640
ROI_SIZE     = 250

# ── Eliminación de fondo por muestreo de bordes
def remove_background(img, tolerance_min=30):
    """
    Segmenta la pastilla del fondo muestreando los píxeles del borde de la imagen.
    Devuelve una máscara binaria (255 = pastilla, 0 = fondo).
    Funciona con cualquier color de fondo uniforme.
    """
    h, w = img.shape[:2]
    borde = max(5, min(h, w) // 20)

    pixeles_borde = np.concatenate([
        img[:borde,  :      ].reshape(-1, 3),
        img[-borde:, :      ].reshape(-1, 3),
        img[:,       :borde ].reshape(-1, 3),
        img[:,       -borde:].reshape(-1, 3),
    ])
    bg_media = np.mean(pixeles_borde, axis=0)
    bg_std   = np.std(pixeles_borde,  axis=0)
    tolerancia = np.maximum(bg_std * 2.5, tolerance_min)

    diff = np.abs(img.astype(np.float32) - bg_media)
    mascara_fondo    = np.all(diff < tolerancia, axis=2).astype(np.uint8) * 255
    mascara_pastilla = cv2.bitwise_not(mascara_fondo)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mascara_pastilla = cv2.morphologyEx(mascara_pastilla, cv2.MORPH_CLOSE, kernel)
    mascara_pastilla = cv2.morphologyEx(mascara_pastilla, cv2.MORPH_OPEN,  kernel)
    return mascara_pastilla


# ── Histogramas HSV normalizados ──────────────────────────────────────────────
def calcular_histogramas(img_bgr, mascara):
    hsv    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    h_hist = cv2.calcHist([hsv], [0], mascara, [180], [0, 180])
    s_hist = cv2.calcHist([hsv], [1], mascara, [256], [0, 256])
    v_hist = cv2.calcHist([hsv], [2], mascara, [256], [0, 256])
    for hist in (h_hist, s_hist, v_hist):
        cv2.normalize(hist, hist, 0, 1, cv2.NORM_MINMAX)
    return h_hist, s_hist, v_hist


# ── Descriptores de forma ─────────────────────────────────────────────────────
def calcular_forma(contorno):
    _, _, w, h = cv2.boundingRect(contorno)
    aspect_ratio = w / h if h > 0 else 1.0
    area      = cv2.contourArea(contorno)
    hull_area = cv2.contourArea(cv2.convexHull(contorno))
    solidity  = area / hull_area if hull_area > 0 else 0.0
    return aspect_ratio, solidity


# ── Calibración ───────────────────────────────────────────────────────────────
pill_profiles = {}

rutas = sorted([
    p for p in glob.glob(f'{PILLS_FOLDER}/*.*')
    if p.lower().endswith(('.jpg', '.jpeg', '.png'))
])

print(f'Calibrando con {len(rutas)} imágenes de referencia...\n')

for ruta in rutas:
    img = cv2.imread(ruta)
    if img is None:
        print(f'  [ERROR] No se pudo leer: {ruta}')
        continue

    # Redimensionar manteniendo proporción
    escala = DISPLAY_SIZE / max(img.shape[:2])
    img = cv2.resize(
        img,
        (int(img.shape[1] * escala), int(img.shape[0] * escala)),
        interpolation=cv2.INTER_AREA
    )

    mascara   = remove_background(img)
    contornos, _ = cv2.findContours(mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contornos:
        print(f'  [AVISO] Sin contorno: {os.path.basename(ruta)}')
        continue

    cnt = max(contornos, key=cv2.contourArea)
    if cv2.contourArea(cnt) < 800:
        print(f'  [AVISO] Contorno muy pequeño: {os.path.basename(ruta)}')
        continue

    mascara_cnt = np.zeros(img.shape[:2], dtype=np.uint8)
    cv2.drawContours(mascara_cnt, [cnt], -1, 255, -1)

    h_hist, s_hist, v_hist = calcular_histogramas(img, mascara_cnt)
    ar, sol = calcular_forma(cnt)

    nombre = os.path.splitext(os.path.basename(ruta))[0]
    pill_profiles[nombre] = {
        'hist_h': h_hist, 'hist_s': s_hist, 'hist_v': v_hist,
        'aspect_ratio': ar, 'solidity': sol,
    }
    print(f'  OK  {nombre:<35s}  AR={ar:.2f}  Solidez={sol:.2f}')

print(f'\n{"-"*50}')
print(f'{len(pill_profiles)} perfiles calibrados correctamente.')

Calibrando con 10 imágenes de referencia...

  OK  317220267                            AR=1.00  Solidez=0.99
  OK  60429-203_M_LH3                      AR=0.97  Solidez=0.73
  OK  634810623                            AR=1.02  Solidez=0.99
  OK  blackNyellow_capsule                 AR=2.92  Solidez=0.98
  OK  bloe_oval                            AR=35.56  Solidez=0.55
  OK  brown_oval                           AR=1.60  Solidez=0.99
  OK  cream_capsule                        AR=1.54  Solidez=0.99
  OK  green_capsule                        AR=2.70  Solidez=0.99
  OK  green_oval                           AR=4.49  Solidez=0.59
  OK  orangeandblue                        AR=2.83  Solidez=0.99

--------------------------------------------------
10 perfiles calibrados correctamente.


## Fase 2 — Detección en cámara

- Abre la webcam (índice `0`; cámbialo a `1`, `2`… si tienes varias).
- Recuadro amarillo = área de análisis. **Coloca la pastilla dentro del recuadro**.
- El fondo del área debe ser contrastante con la pastilla (mesa oscura para pastillas blancas, etc.).
- Cuando el recuadro se pone **verde** hay una coincidencia con score ≥ `UMBRAL_MATCH`.
- Presiona **`q`** para cerrar la cámara.

In [35]:
if not pill_profiles:
    print('No hay perfiles. Ejecuta primero la celda de calibración.')
else:
    INDICE_CAMARA = 1     # Cambiar si la webcam correcta tiene otro índice
    UMBRAL_MATCH  = 0.55  # Score mínimo para aceptar coincidencia (rango 0–1)
    AREA_MINIMA   = 1500  # Píxeles mínimos de pastilla en el ROI

    # Pesos para el score de color (H=tono, S=saturación, V=brillo)
    PESO_H     = 0.50
    PESO_S     = 0.30
    PESO_V     = 0.20
    # Peso color vs forma en el score final
    PESO_COLOR = 0.70
    PESO_FORMA = 0.30

    cap = cv2.VideoCapture(INDICE_CAMARA)
    if not cap.isOpened():
        print(f'[ERROR] No se pudo abrir la cámara {INDICE_CAMARA}.')
    else:
        print('Camara lista. Coloca una pastilla en el recuadro amarillo.')
        print('Presiona "q" para salir.\n')

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            fh, fw = frame.shape[:2]
            rx1 = fw // 2 - ROI_SIZE // 2
            ry1 = fh // 2 - ROI_SIZE // 2
            rx2 = rx1 + ROI_SIZE
            ry2 = ry1 + ROI_SIZE
            roi = frame[ry1:ry2, rx1:rx2].copy()

            # Segmentar pastilla en el ROI
            mascara_roi = remove_background(roi, tolerance_min=25)
            cnts_roi, _ = cv2.findContours(
                mascara_roi, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
            )

            mejor_nombre = None
            mejor_score  = 0.0

            if cnts_roi:
                cnt_roi = max(cnts_roi, key=cv2.contourArea)

                if cv2.contourArea(cnt_roi) >= AREA_MINIMA:
                    # Máscara del contorno detectado
                    m_cnt = np.zeros(roi.shape[:2], dtype=np.uint8)
                    cv2.drawContours(m_cnt, [cnt_roi], -1, 255, -1)

                    # Histogramas y forma de la consulta
                    qh, qs, qv = calcular_histogramas(roi, m_cnt)
                    ar_q, sol_q = calcular_forma(cnt_roi)

                    # Comparar con cada perfil calibrado
                    for nombre, perfil in pill_profiles.items():
                        sc_h = cv2.compareHist(qh, perfil['hist_h'], cv2.HISTCMP_CORREL)
                        sc_s = cv2.compareHist(qs, perfil['hist_s'], cv2.HISTCMP_CORREL)
                        sc_v = cv2.compareHist(qv, perfil['hist_v'], cv2.HISTCMP_CORREL)
                        score_color = sc_h * PESO_H + sc_s * PESO_S + sc_v * PESO_V

                        ar_sim  = 1.0 - min(
                            abs(ar_q  - perfil['aspect_ratio']) / max(perfil['aspect_ratio'], 0.1),
                            1.0
                        )
                        sol_sim = 1.0 - min(abs(sol_q - perfil['solidity']), 1.0)
                        score_forma = ar_sim * 0.6 + sol_sim * 0.4

                        score_total = score_color * PESO_COLOR + score_forma * PESO_FORMA

                        if score_total > mejor_score:
                            mejor_score  = score_total
                            mejor_nombre = nombre

            # ── Dibujar resultados ────────────────────────────────────────────
            hay_match  = mejor_nombre is not None and mejor_score >= UMBRAL_MATCH
            color_rect = (0, 255, 0) if hay_match else (0, 255, 255)

            cv2.rectangle(frame, (rx1, ry1), (rx2, ry2), color_rect, 2)
            cv2.putText(
                frame, 'Area de analisis', (rx1, ry1 - 8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color_rect, 1
            )

            if hay_match:
                texto = f'Detectado: {mejor_nombre}  ({mejor_score:.2f})'
                cv2.putText(frame, texto, (20, 38),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 0), 2)
            else:
                extra = f'  (candidato: {mejor_nombre} {mejor_score:.2f})' if mejor_nombre else ''
                cv2.putText(frame, f'Sin coincidencia{extra}', (20, 38),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.60, (0, 0, 255), 2)

            cv2.putText(
                frame, 'Presiona "q" para salir', (20, fh - 12),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (180, 180, 180), 1
            )
            cv2.imshow('Drug Wizard - Detector en Camara', frame)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()
        print('Camara cerrada.')

Camara lista. Coloca una pastilla en el recuadro amarillo.
Presiona "q" para salir.

Camara cerrada.
